In [199]:
import os
from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import  ChatHuggingFace,HuggingFaceEndpoint
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

load_dotenv()

True

# Indexing(document ingestion)

In [200]:
video_id = "-HzgcbRXUK8"

try:
    ytt_api = YouTubeTranscriptApi()
    transcript_data = ytt_api.fetch(video_id, languages=['en'])

    # transcript_data = YouTubeTranscriptApi.get_transcript(video_id, languages=['en'])
    transcript = " ".join(snippet.text for snippet in transcript_data)
    print(transcript)
except TranscriptsDisabled:
    print("No captions available for this video.")


- It's hard for us humans to make any kind of clean predictions about highly nonlinear, dynamical systems. But again, to your point, we might be very surprised
what classical learning systems might be able to do about even fluid. - Yes, exactly. I mean, fluid dynamics,
Navier-Stokes equations, these are traditionally thought of as very, very difficult intractable problems to do on classical systems. They take enormous amounts of compute, you know, weather prediction systems, you know, these kind of things all involve fluid dynamics calculations. But again, if you look
at something like Veo, our video generation model, it can model liquids quite
well, surprisingly well, and materials, specular lighting. I love the ones where, you know, there's people who generated videos where there's like clear liquids going through hydraulic presses, and then it's being squeezed out. I used to write physics
engines and graphics engines in my early days in gaming, and I know it's just so painstakingly 

# Text Splitting

In [201]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap = 200)
chunks = splitter.create_documents([transcript])
len(chunks)

185

Embedding and Vector Store

In [202]:
embedding = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
)
vector_store = FAISS.from_documents(chunks,embedding)

In [203]:
vector_store.index_to_docstore_id

{0: 'a60a2a90-ae5b-46dd-bacf-2ed0d5bf9544',
 1: 'f84c524b-c947-453e-8918-b078e8078489',
 2: '80297c86-8882-4538-ba79-2042a0889615',
 3: '521080ba-02f3-4c0e-a8d1-118a9e866408',
 4: '8cc1f9e0-08f1-493b-ac4d-848e75a7f6ee',
 5: '6e05d372-c62e-4a51-bffe-ca2abffc891b',
 6: '201c2e87-a10b-4daa-949f-f2c523070bb5',
 7: '31a7d10a-8d1a-4313-b76c-9b1418481d91',
 8: '455abe22-7825-4a70-9df7-a4aa3739c7d1',
 9: '9ca4057c-a7ef-498e-a8eb-47e26aa4f825',
 10: 'dacbbd90-0643-482d-b4ac-8c71f9ea4827',
 11: 'bdc36d0c-b1ea-41e7-b563-4334ce705c9b',
 12: 'cc057e86-0e18-4199-ae83-5d6a6103c373',
 13: '007b6fbe-905c-4413-8eeb-52b559130a86',
 14: 'e9dd60c8-0887-4eb1-b24f-607cc7dad7ea',
 15: '2ce93ccb-a62f-4c6f-8ed5-048618777f5a',
 16: 'f99fd115-cfbc-4fe2-94b1-e276bc6483ce',
 17: '2ee82e28-ec9b-460c-a381-d2139a091d82',
 18: 'f0b5cf54-f995-402d-89fa-26a7bd1f217f',
 19: 'cf286048-8e43-41a5-85ad-186919bb756c',
 20: 'd51a5b49-b9a9-4614-8477-5f1f1744215c',
 21: 'fd56d27b-d38e-4a38-b542-652ae092fc4b',
 22: 'd8a6890e-a6ef-

# Retriever

In [204]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":4})

In [205]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000017FD394C9F0>, search_kwargs={'k': 4})

In [206]:
retriever.invoke("What is deepmind")

[Document(id='8a9f034d-75d0-464c-8f03-f74e5d99072b', metadata={}, page_content="of like any big company, you know, ends up having a\nlot of layers of management and things like that is sort\nof the nature of how it works. But I still operate and\nI was always operating with old DeepMind as a startup still. A large one, but still as a startup. And that's what we still act like today as with Google DeepMind. And acting with decisiveness\nand the energy that you get from the best smaller organizations. And we try to get the best of both worlds where we have this incredible\nbillions of users surfaces, incredible products that we can power up with our AI and our research. And that's amazing. And you can, you know, there's very few places in\nthe world you can get that, do incredible world-class\nresearch on the one hand and then plug it in and improve billions of\npeople's lives the next day. That's a pretty amazing combination. And we're continually fighting\nand cutting away bureaucracy 

# Augumentation

In [207]:
llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="conversational",  # Add this line
    max_new_tokens=512,
    do_sample=True,
    temperature=0.1,
    repetition_penalty=1.03,
)

In [208]:
prompt = PromptTemplate(
    template = """You are a helpful assistant.
    Answer only from provided transcript content.
    If content is insufficinet, just say you don't know
    {context}
    Question: {question}
    """,
    input_variables=['context','question']
)

In [209]:
question = "Is the topic of Neural Network discussed in this video, if yes what was it ?"
retireved_docs = retriever.invoke(question)

In [210]:
context_docs = "\n\n".join(doc.page_content for doc in retireved_docs)

In [211]:
final_prompt = prompt.invoke({'context':context_docs, 'question':question})

In [212]:
final_prompt

StringPromptValue(text="You are a helpful assistant.\n    Answer only from provided transcript content.\n    If content is insufficinet, just say you don't know\n    science kind of way? - Yeah, I think that there are actually a huge class of problems that could be couched in this way, the way we did AlphaGo and\nthe way we did AlphaFold, where, you know, you model what the\ndynamics of the system is, the properties of that system, the environment that you\nare trying to understand. And then that makes the\nsearch for the solution or the prediction of\nthe next step efficient basically polynomial times, so tractable by a classical system, which a neural network is. It runs on normal computers, right, classical computers,\nTuring machines in effect. And I think it's one of the most\ninteresting questions there is is how far can that paradigm go? You know, I think we've proven\nthe AI community in general that classical systems, Turing\nmachines can go a lot further than we previously th

# Generation

In [213]:

result = llm.invoke(final_prompt)
print(result)

ValueError: Model mistralai/Mistral-7B-Instruct-v0.2 is not supported for task text-generation and provider featherless-ai. Supported task: conversational.